# Pipeline smoke test

Raw Citi Bike data -> `RawModelData` -> `ResolvedModelData` -> `Environment` -> `SimulationLog`.

The base scenario reproduces historical **demand** exactly. `FormDeparturesPhase`
re-releases every historical departure (gated by stock), and
`FormPotentialTripsPhase` assigns each departure a destination and duration from
the OD demand model `P(target | source, commodity)` + mean historical duration.

To make the replay exact, the resolved data is built with `saturate_stock=True`:
artificial saturated stock and dock capacities replace the GBFS snapshot, so the
demand gate and the overflow redirect stay in the pipeline but never bind. (The
GBFS snapshot is a current observation, unrelated to the historical start state,
and would starve the replay with stockouts that never happened.)

Because targets and durations come from the (aggregate) OD model rather than each
trip's own record, the per-trip journal is **not** identical to history, and the
OD-count matrix drifts slightly under per-period largest-remainder rounding. The
marginal that is preserved exactly is the departures table:
`simulated_departures_df == historical_departures_df`.

In [1]:
import pandas as pd

from gbp.loaders.dataloader_raw import RawModelData
from gbp.loaders.dataloader_graph import ResolvedModelData, attach_simulation
from gbp.consumers.simulator.engine import Environment, EnvironmentConfig
from gbp.consumers.simulator import (
    ArrivalsPreviousPhase,
    OverflowRedirectPreviousPhase,
    FormDeparturesPhase,
    FormPotentialTripsPhase,
    ArrivalsPhase,
    OverflowRedirectPhase,
)

In [ ]:
# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path="../data/raw/202602-citibike-tripdata_1.csv",
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

# Graph (resolved) model data: period grid, historical flows, replay demand.
# saturate_stock=True swaps the GBFS snapshot for artificial saturated stock and
# capacities, so demand gating and overflow redirect stay in the pipeline but
# never bind -- the base replay reproduces the historical departures exactly.
graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1), saturate_stock=True)

historical_flows_df = graph_data.historical_flows_df

In [ ]:
phases_canonical = [
    ArrivalsPreviousPhase(),
    OverflowRedirectPreviousPhase(),
    FormDeparturesPhase(),
    FormPotentialTripsPhase(),
    ArrivalsPhase(),
    OverflowRedirectPhase(),
]

env_canonical = Environment(
    graph_data,
    EnvironmentConfig(phases=phases_canonical, seed=42, scenario_id="historical_replay"),
)
env_canonical.run()

# Wire the finished run back into the graph-data container's simulated_* slots.
attach_simulation(graph_data, env_canonical.simulated_flows_df)
simulated_flows_df = graph_data.simulated_flows_df
simulated_departures_df = graph_data.simulated_departures_df
historical_departures_df = graph_data.historical_departures_df


def _sorted(df):
    return (df.sort_values(["period_id", "facility_id", "commodity_category"])
            .reset_index(drop=True))


# The base scenario reproduces historical demand exactly: the departures marginal
# of the simulated journal equals the historical one. (The per-trip journal is not
# identical, because targets/durations are drawn from the aggregate OD model.)
pd.testing.assert_frame_equal(_sorted(simulated_departures_df), _sorted(historical_departures_df))
print("simulated_departures_df == historical_departures_df:",
      _sorted(simulated_departures_df).equals(_sorted(historical_departures_df)))